<a href="https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/OpenEnv_gpt_oss_(20B)_Reinforcement_Learning_2048_Game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="打开方式科拉布”/></a>

# 目标：让gpt-oss用强化学习玩游戏

我们的目标是让OpenAI的开放模型gpt-oss 20b通过强化学习来玩2048游戏。我们希望模型设计一个玩 2048 的策略，我们将运行这个策略，直到我们赢或输。

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/f9/2048_win.png/500px-2048_win.png" height=300 />

# 安装
我们将使用 [Unsloth](https://github.com/unslothai/unsloth) 在 GPT-OSS 20B 上进行强化学习，并使用 [OpenEnv](https://github.com/meta-pytorch/OpenEnv) 进行环境交互。 Unsloth 节省了 70% 的 VRAM 使用量，并使强化学习速度提高 2 到 6 倍！

In [1]:
%%bash
python -m pip install -qU uv --root-user-action=ignore

ROCM_TAG="$({ command -v amd-smi >/dev/null 2>&1 && amd-smi version 2>/dev/null | awk -F'ROCm version: ' 'NF>1{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { [ -r /opt/rocm/.info/version ] && awk -F. '{print "rocm"$1"."$2; exit}' /opt/rocm/.info/version; } || { command -v hipconfig >/dev/null 2>&1 && hipconfig --version 2>/dev/null | awk -F': *' '/HIP version/{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { command -v dpkg-query >/dev/null 2>&1 && ver="$(dpkg-query -W -f='${Version}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; } || { command -v rpm >/dev/null 2>&1 && ver="$(rpm -q --qf '%{VERSION}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; })"
[ -n "$ROCM_TAG" ] || { echo "Could not detect ROCm. Install ROCm first or set ROCM_TAG manually."; exit 1; }
case "$ROCM_TAG" in
  rocm6.[0-4]|rocm7.[02]) T="$ROCM_TAG" ;;
  rocm6.*) T="rocm6.4" ;;
  *) T="rocm7.1" ;;
esac
pip install bitsandbytes
PYTORCH_INDEX_URL="https://download.pytorch.org/whl/${T}"
uv pip install --system -U --force-reinstall \
    torch torchvision torchaudio triton-rocm \
    --index-url "$PYTORCH_INDEX_URL"
uv pip install --system cut-cross-entropy torchao --no-deps
uv pip install --system -U --no-deps "unsloth[amd]" "unsloth_zoo[amd]"
uv pip install --system --no-deps -r "$(python -c 'import pathlib,site;print(next(p for r in [*site.getsitepackages(),site.getusersitepackages()] if (p:=pathlib.Path(r,"studio/backend/requirements/no-torch-runtime.txt")).exists()))')" torchao
uv pip install --system --no-deps -U "tokenizers>=0.22.0,<=0.23.0"


In [ ]:
!uv pip install --system -qqq "transformers==4.56.2" trackio
!uv pip install --system -qqq --upgrade --no-deps "trl==0.22.2"


然后我们将从源安装 [OpenEnv](https://github.com/meta-pytorch/OpenEnv)：

In [3]:
%%capture
!pip install -qqq fastapi uvicorn requests open_spiel
!pip install fastapi uvicorn requests
!pip install open_spiel --prefer-binary
!git clone https://github.com/meta-pytorch/OpenEnv.git > /dev/null 2>&1
%cd OpenEnv
!git checkout 83dda10
import subprocess, sys, os
from pathlib import Path
sys.path.insert(0, '.')  # 为 envs 模块添加 OpenEnv 根目录
sys.path.insert(0, './src')
working_directory = str(Path.cwd().parent.absolute() / "OpenEnv")

我们将加载 GPT-OSS 20B 并设置一些参数：
* `max_seq_length = 768` 模型的最大上下文长度。增加它会使用更多内存。
* `lora_rank = 4` 这个数字越大，RL 过程越智能，但速度越慢且占用内存越多`load_in_16bit` 会更快，但需要 64GB 或更多 GPU (MI300)
* `offload_embedding = True` 新的 Unsloth 优化将嵌入移至 CPU RAM，从而将 VRAM 减少 1GB。

In [4]:
import os
from unsloth import FastLanguageModel
import torch
max_seq_length = 768 # 可以增加更长的 RL 输出
lora_rank = 4        # 排名越高=越聪明，但速度越慢
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    load_in_4bit = True,
    max_seq_length = max_seq_length,
    offload_embedding = True, # 卸载嵌入以节省更多 VRAM
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.3: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.37G [00:00<?, ?B/s]

...

Unsloth: Offloading embeddings to RAM to save 1.08 GB.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

为了进行高效的强化学习，我们将使用 [LoRA](https://arxiv.org/abs/2106.09685)，它允许我们仅向模型添加 1 到 5% 的额外权重以进行微调。这使我们能够节省 60% 以上的内存使用量，同时保持良好的准确性。阅读 Unsloth 的 [GPT-OSS RL Guide](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning) 了解更多详细信息。

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # 选择任意大于 0 的数字！建议 8、16、32、64、128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 加快训练速度
    use_gradient_checkpointing = "unsloth", # 减少内存使用
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model` require gradients


# 使用 OpenEnv 的 2048 游戏环境

我们首先启动一个 OpenEnv 进程并导入它！这将使我们能够看到 2048 的实现是什么样子的！

In [6]:
from envs.openspiel_env import OpenSpielEnv
from envs.openspiel_env.models import OpenSpielAction, OpenSpielObservation

我们将使用 Unsloth 的 OpenEnv 实现，并使用一些设置参数包装 `launch_openenv`：

In [7]:
global port
global openenv_process
port = 9000
openenv_process = None
server = "envs.openspiel_env.server.app:app"
environment = {
    **os.environ,
    "PYTHONPATH": f"{working_directory}/src",
    "OPENSPIEL_GAME": "2048",
    "OPENSPIEL_AGENT_PLAYER": "0",
    "OPENSPIEL_OPPONENT_POLICY": "random",
}

# 增强Unsloth的OpenEnv创建功能
import functools
from unsloth import is_port_open, launch_openenv
launch_openenv = functools.partial(
    launch_openenv,
    working_directory = working_directory,
    server = server,
    environment = environment,
    openenv_class = OpenSpielEnv,
)

让我们看看当前的2048游戏状态是什么样的：

In [8]:
port, openenv_process = launch_openenv(port, openenv_process)
result = openenv_process.reset()
current_state = result.observation
current_state

Unsloth: Creating new OpenEnv process at port = 12724.....................


OpenSpielObservation(done=False, reward=None, metadata={}, info_state=[0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], legal_actions=[0, 1, 2, 3], game_phase='initial', current_player_id=0, opponent_last_action=None)

首先让我们将状态转换为数字列表！

In [9]:
import numpy as np
def convert_to_board(current_state):
    n = len(current_state.info_state)
    size = int(np.sqrt(n))
    board = np.array_split(np.array(current_state.info_state, dtype = int), size)
    board = [x.tolist() for x in board]
    return board, size
convert_to_board(current_state)

([[0, 0, 0, 2], [0, 0, 2, 0], [0, 0, 0, 0], [0, 0, 0, 0]], 4)

我们还想漂亮地打印游戏板！

In [10]:
# @title（可折叠）2048 游戏渲染器
def render_board(obs, colors: bool = True, border: bool = True, dot_for_zero: bool = True) -> str:
    """
    Pretty-print the board with colors that scale from 0 up to self.target.
    Uses ANSI 256-color codes (works in most terminals). Set colors = False to disable.
    """
    import math
    b, size = convert_to_board(obs)
    mx = max((max(row) for row in b), default = 0)
    cell_w = max(3, len(str(mx)))

    RESET = "\x1b[0m"

    # 从冷色→暖色的平滑渐变
    # （蓝色/青色/绿色→黄色/橙色/红色）。根据需要调整或扩展。
    GRAD = [33, 39, 45, 51, 50, 49, 48, 47, 46, 82, 118, 154, 190, 226, 220, 214, 208, 202, 196]
    ZERO_FG = 239  # 暗灰色

    def color_code(v: int) -> str:
        if not colors:
            return ""
        if v == 0:
            return f"\x1b[38;5;{ZERO_FG}m"
        # 按相对于目标的指数进行标准化：[0,1] 中的 r
        t = max(2, 2048)  # 安全;避免 log2(1)
        # Guard：如果 v 不是 2 的幂或者 <1，则优雅地处理
        try:
            r = max(0.0, min(1.0, math.log2(v) / math.log2(t)))
        except ValueError:
            r = 0.0
        idx = int(round(r * (len(GRAD) - 1)))
        return f"\x1b[38;5;{GRAD[idx]}m"

    def fmt(v: int) -> str:
        s = "." if (v == 0 and dot_for_zero) else str(v)
        s = s.rjust(cell_w)
        return color_code(v) + s + (RESET if colors else "")

    def hline(left: str, mid: str, right: str) -> str:
        return left + mid.join("─" * cell_w for _ in range(size)) + right

    rows = []
    if border:
        rows.append(hline("┌", "┬", "┐"))
    for r in range(size):
        content = "│".join(fmt(v) for v in b[r])
        rows.append(("│" + content + "│") if border else content)
        if border:
            rows.append(hline("└" if r == size - 1 else "├",
                            "┴" if r == size - 1 else "┼",
                            "┘" if r == size - 1 else "┤"))
    return "\n".join(rows)

In [11]:
print(render_board(current_state))

┌───┬───┬───┬───┐
│  .│  .│  .│  2│
├───┼───┼───┼───┤
│  .│  .│  2│  .│
├───┼───┼───┼───┤
│  .│  .│  .│  .│
├───┼───┼───┼───┤
│  .│  .│  .│  .│
└───┴───┴───┴───┘


我们可以看到“legal_actions”，即您可以将其视为“[0,1,2,3]”让我们尝试执行操作“0”。

In [12]:
action = OpenSpielAction(action_id = 0, game_name = "2048")
result = openenv_process.step(action)
current_state = result.observation
print(render_board(current_state))

┌───┬───┬───┬───┐
│  .│  .│  2│  2│
├───┼───┼───┼───┤
│  .│  .│  .│  .│
├───┼───┼───┼───┤
│  .│  .│  .│  .│
├───┼───┼───┼───┤
│  .│  .│  2│  .│
└───┴───┴───┴───┘


所以看起来“0”是一个向上移动的动作！让我们尝试一下“1”。

In [13]:
action = OpenSpielAction(action_id = 1, game_name = "2048")
result = openenv_process.step(action)
current_state = result.observation
print(render_board(current_state))

┌───┬───┬───┬───┐
│  .│  .│  .│  4│
├───┼───┼───┼───┤
│  2│  .│  .│  .│
├───┼───┼───┼───┤
│  .│  .│  .│  .│
├───┼───┼───┼───┤
│  .│  .│  .│  2│
└───┴───┴───┴───┘


“1”是向右移动的动作。和“2”：

In [14]:
action = OpenSpielAction(action_id = 2, game_name = "2048")
result = openenv_process.step(action)
current_state = result.observation
print(render_board(current_state))

┌───┬───┬───┬───┐
│  .│  .│  .│  .│
├───┼───┼───┼───┤
│  .│  .│  .│  .│
├───┼───┼───┼───┤
│  .│  .│  .│  4│
├───┼───┼───┼───┤
│  2│  2│  .│  2│
└───┴───┴───┴───┘


“2”是向下移动。我猜‘3’就是向左移动！

In [15]:
action = OpenSpielAction(action_id = 3, game_name = "2048")
result = openenv_process.step(action)
current_state = result.observation
print(render_board(current_state))

┌───┬───┬───┬───┐
│  .│  .│  .│  .│
├───┼───┼───┼───┤
│  .│  .│  .│  .│
├───┼───┼───┼───┤
│  4│  .│  .│  .│
├───┼───┼───┼───┤
│  4│  2│  .│  2│
└───┴───┴───┴───┘


我们还可以打印游戏状态，指示是否无法进行更多操作，以及您可以采取的可能操作！

In [16]:
print(current_state.done)
print(current_state.legal_actions)

False
[0, 1, 3]


# 强化学习环境设置

我们将设置一个函数来接受一些策略，该策略将在“0123”内发出一个动作并检查游戏状态。

我们还将添加一个计时器，最多只执行该策略 2 秒，否则它可能永远不会终止！

In [17]:
from typing import Callable
from unsloth import execute_with_time_limit
import itertools

def _execute_strategy(strategy, current_state : OpenSpielObservation):
    assert callable(strategy)

    steps = 0
    total_reward = 0
    while not current_state.done:
        board, size = convert_to_board(current_state)
        action = strategy(board)
        try:
            action = int(action)
        except:
            return steps, False
        steps += 1
        if type(action) is not int or action not in current_state.legal_actions:
            return steps, max(itertools.chain.from_iterable(board)) == 2048

        global port, openenv_process
        port, openenv_process = launch_openenv(port, openenv_process)
        action = OpenSpielAction(action_id = action, game_name = "2048")
        result = openenv_process.step(action)
        current_state = result.observation
        if result.reward is not None:
            total_reward += result.reward
    return steps, max(itertools.chain.from_iterable(board)) == 2048

@execute_with_time_limit(2)
def execute_strategy(strategy : Callable, current_state : OpenSpielObservation):
    return _execute_strategy(strategy, current_state)

让我们制定一个通用策略来点击“3”。我们应该预料到这种通用策略会失败：

In [18]:
def always_move_left(board):
    return 3

# 将 OpenEnv 重置为初始状态！
port, openenv_process = launch_openenv(port, openenv_process)
result = openenv_process.reset()
current_state = result.observation
try:
    steps, if_done = execute_strategy(always_move_left, current_state)
except TimeoutError as e:
    print(f"Timed out with error = {str(e)}")

steps, if_done

(3, False)

为了允许 GPT-OSS 强化学习使用更长的策略，我们将允许 5 秒的计时器。

In [19]:
@execute_with_time_limit(5)
def execute_strategy(strategy : Callable, current_state : OpenSpielObservation):
    return _execute_strategy(strategy, current_state)

# 代码执行

要执行并创建一个新的Python函数，我们首先必须检查该函数是否没有调用其他全局变量或作弊。这称为“对抗奖励黑客”，因为我们不希望该函数作弊。

例如，下面的代码就很好，因为它只导入 Python 级别的函数。我们使用“check_python_modules”：

In [20]:
from unsloth import check_python_modules

sample = """
def strategy(board):
    import math
    from typing import Callable
    return "0"
"""
ok, info = check_python_modules(sample)
print("只导入Python？", ok)
print(info)

Only Python imports? True
{'stdlib': ['math', 'typing'], 'non_stdlib': [], 'relative_imports': 0}


对于下面的代码，由于我们导入了“numpy”，所以我们不应该允许执行：

In [21]:
sample = """
def strategy(board):
    from numpy import matmul
    return "0"
"""
ok, info = check_python_modules(sample)
print("只导入Python？", ok)
print(info)

Only Python imports? False
{'stdlib': [], 'non_stdlib': ['numpy'], 'relative_imports': 0}


我们也不允许全局变量访问。我们将使用 Unsloth 的 `create_locked_down_function` 函数

In [22]:
from unsloth import create_locked_down_function
function = """
def import_numpy():
    np.matmul
    print("成功")
"""
f = create_locked_down_function(function)
try:
    f()
except Exception as e:
    print(str(e))

name 'np' is not defined


In [23]:
from unsloth import create_locked_down_function
function = """
def add(a, b):
    def adder(a):
        return a + b
    return adder(b) + b
"""
f = create_locked_down_function(function)
try:
    print(f(10, 20))
except Exception as e:
    print(str(e))

60


# 数据和强化学习任务设置

我们现在必须创建一个提示，告诉模型为 2048 游戏创建策略。您可以将其自定义为其他 RL 任务的其他任务。

In [24]:
prompt = """
Create a new short 2048 strategy using only native Python code.
You are given a list of list of numbers for the current board state.
Output one action for "0", "1", "2", "3" on what is the optimal next step.
Output your new short function in backticks using the format below:
```python
def strategy(board):
    return "0" # 例子
```
All helper functions should be inside def strategy. Only output the short function `strategy`.
""".strip()
print(prompt)

Create a new short 2048 strategy using only native Python code.
You are given a list of list of numbers for the current board state.
Output one action for "0", "1", "2", "3" on what is the optimal next step.
Output your new short function in backticks using the format below:
```python
def strategy(board):
    return "0" # Example
```
All helper functions should be inside def strategy. Only output the short function `strategy`.


首先，我们来提示一下没有 RL 的 GPT-OSS，看看效果如何：

In [25]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize = False,
    add_generation_prompt = True,
    reasoning_effort = "low",
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1.0,
    max_new_tokens = 512,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-25

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Create a new short 2048 strategy using only native Python code.
You are given a list of list of numbers for the current board state.
Output one action for "0", "1", "2", "3" on what is the optimal next step.
Output your new short function in backticks using the format below:
```python
def strategy(board):
    return "0" # Example
```
All helper functions should be inside def strategy. Only output the short function `strategy`.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to provide a short function. Probably simple heuristic: choose move with lowest collision? Use sum? Just a placeholder.<|end|><|start|>assistant<|channel|>final<|me

# 奖励功能

我们现在设计一个“extract_function”函数，它简单地提取包含在 3 个反引号中的函数。

以及 3 个奖励函数：

1. `function_works` 如果策略是有效的 Python 函数，则会奖励模型。
2. `no_cheating` 检查函数是否导入了其他模块，如果导入了，我们就会对其进行惩罚。
3. `strategy_succeeds` 检查运行自动生成的策略后游戏策略是否确实成功达到 2048。

In [26]:
def extract_function(text):
    if text.count("```") >= 2:
        first = text.find("```") + 3
        second = text.find("```", first)
        fx = text[first : second].strip()
        fx = fx.removeprefix("python\n")
        fx = fx[fx.find("def"):]
        if fx.startswith("def strategy(board):"): return fx
    return None
print(extract_function(prompt))

def strategy(board):
    return "0" # Example


下面是我们的“function_works”奖励函数，它使用Python的“exec”，但通过不允许泄漏本地和全局变量来保护。我们还可以在执行函数之前先使用 check_python_modules 来检查是否有错误：

In [27]:
ok, info = check_python_modules("def a")
ok, info

(False,
 {'error': "SyntaxError: expected '(' (<unknown>, line 1)",
  'stdlib': [],
  'non_stdlib': [],
  'relative_imports': 0})

In [28]:
def function_works(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        function = extract_function(response)
        if function is not None:
            ok, info = check_python_modules(function)
        if function is None or "error" in info:
            score = -2.0
        else:
            try:
                new_strategy = create_locked_down_function(function)
                score = 1.0
            except:
                score = -0.5
        scores.append(score)
    return scores

`no_cheating` 检查函数是否作弊，因为它可能导入了 Numpy 或其他函数：

In [29]:
def no_cheating(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        function = extract_function(response)
        if function is not None:
            ok, info = check_python_modules(function)
            scores.append(1.0 if ok else -20.0) # 重罚！
        else:
            scores.append(-1.0) # 创建函数失败
    return scores

接下来“strategy_succeeds”检查该策略是否确实允许游戏终止。想象一下，如果该策略只是返回“0”，那么它会在 10 秒的时间限制后失败。

我们还添加了一个全局“打印机”来打印策略和棋盘状态。

In [30]:
import numpy as np
global PRINTER
PRINTER = 0
def strategy_succeeds(completions, **kwargs):
    global PRINTER
    scores = []
    for completion in completions:
        printed = False
        score = 0
        response = completion[0]["content"]
        function = extract_function(response)
        if PRINTER % 5 == 0:
            printed = True
            print(function)
        PRINTER += 1
        if function is not None:
            ok, info = check_python_modules(function)
        if function is None or "error" in info:
            scores.append(0)
            continue
        try:
            new_strategy = create_locked_down_function(function)
        except:
            scores.append(0)
            continue
        try:
            # 将 OpenEnv 重置为初始状态！
            global port, openenv_process
            port, openenv_process = launch_openenv(port, openenv_process)
            result = openenv_process.reset()
            current_state = result.observation
            steps, if_done = execute_strategy(new_strategy, current_state)
            print(f"Steps = {steps} If Done = {if_done}")
            if printed is False:
                print(function)
            print(render_board(current_state))
            if if_done:
                scores.append(20.0) # 成功——海量奖励！
            else:
                scores.append(2.0) # 失败但功能有效！
        except TimeoutError as e:
            print("暂停")
            scores.append(-1.0) # 超时失败
        except Exception as e:
            print(f"Exception = {str(e)}")
            scores.append(-3.0) # 失败的
    return scores

我们现在将创建包含提示副本的数据集。记得加上低的推理努力！您可以选择高推理模式，但这仅适用于更多内存的 GPU，例如 MI300。

In [31]:
from datasets import Dataset
dataset = Dataset.from_list([{"prompt" : [{"role": "user", "content": prompt.strip()}], "answer" : 0, "reasoning_effort": "low"}]*1000)
maximum_length = len(tokenizer.apply_chat_template([{"role": "user", "content": prompt.strip()}], add_generation_prompt = True))
print(maximum_length)
dataset[0]

181


{'prompt': [{'content': 'Create a new short 2048 strategy using only native Python code.\nYou are given a list of list of numbers for the current board state.\nOutput one action for "0", "1", "2", "3" on what is the optimal next step.\nOutput your new short function in backticks using the format below:\n```python\ndef strategy(board):\n    return "0" # Example\n```\nAll helper functions should be inside def strategy. Only output the short function `strategy`.',
   'role': 'user'}],
 'answer': 0,
 'reasoning_effort': 'low'}

<a name="火车"></a>
### 训练模型

现在设置 GRPO Trainer 和所有配置！我们还支持 GSPO、GAPO、GRPO 博士等！前往 Unsloth [Reinforcement Learning Docs](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) 了解更多选项。

我们还使用 [TrackIO](https://github.com/gradio-app/trackio)，它允许您直接在本地完全本地化笔记本内可视化所有训练指标！

In [32]:
max_prompt_length = maximum_length + 1 # + 1 以防万一！
max_completion_length = max_seq_length - max_prompt_length

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 2e-4,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # 增加到 4 以使训练更顺畅
    num_generations = 2, # 如果内存不足则减少
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # 设置为 1 以进行完整的训练
    max_steps = 600,
    save_steps = 100,
    report_to = "trackio", # 可以使用权重和偏差、TrackIO
    output_dir = "outputs",

    # 用于可选培训+评估
    # fp16_full_eval = 真，
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "步骤",
    # 评估步骤 = 1,
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 2


让我们运行训练器吧！如果向上滚动，您会看到奖励表。目标是看到“奖励”栏增加！

您可能需要等待 150 到 200 步才能执行任何操作。前 100 步您可能会获得 0 奖励。请耐心等待！

|步骤|训练损失|奖励 |奖励标准 |完成长度 |吉隆坡 |
|------|-------------|------------|------------|--------------------|----------|
| 1 | 0.000000 | 0.000000 0.125000 | 0.000000 | 0.000000 200.000000 | 0.000000 | 0.000000
| 2 | 0.000000 | 0.000000 0.072375 | 0.072375 0.248112 | 200.000000 | 0.000000 | 0.000000
| 3 | 0.000000 | 0.000000 -0.079000 | -0.079000 0.163776 | 182.500000 | 0.000005 | 0.000005

In [33]:
# 用于可选培训+评估
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        function_works,
        no_cheating,
        strategy_succeeds,
    ],
    args = training_args,
    train_dataset = dataset,

    # 用于可选培训+评估
    # train_dataset = new_dataset["火车"],
    # eval_dataset = new_dataset["测试"],
)

Unsloth: Switching to float32 training since model cannot work with float16


让我们训练模型吧！ **注意** 这可能会很慢！ 600 步大约需要 5 小时或更长时间。

[TrackIO](https://github.com/gradio-app/trackio) 加载可能有点慢 - 等待 2 分钟，直到图表弹出！

In [34]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 600
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 1,990,656 of 20,916,747,840 (0.01% trained)


* Running on public URL: https://93f7f59fd9e1813cea.gradio.live
* Trackio project initialized: huggingface
* Trackio metrics logged to: /root/.cache/huggingface/trackio


* Created new run: dainty-sunset-0


`generation_config` default values have been modified to match model-specific defaults: {'max_length': 131072}. If this is not desired, please set these values explicitly.


def strategy(board):
    # Simulate each move and choose one maximizing the largest tile after the move
    moves = ["0", "1", "2", "3"]  # 0:left, 1:right, 2:up, 3:down
    best_move, best_max = None, -1
    
    for m in moves:
        nxt = [row[:] for row in board]
        if m == "0":
            for row in nxt:
                l = [x for x in row if x]
                l += [0]*(4-len(l))
                for i in range(4):
                    row[i] = l[i]
        elif m == "1":
            for row in nxt:
                r = [x for x in row if x][::-1]
                r += [0]*(4-len(r))
                for i in range(4):
                    row[i] = r[::-1][i]
        elif m == "2":
            cols = list(zip(*nxt))
            for i, col in enumerate(cols):
                l = [x for x in col if x]
                l += [0]*(4-len(l))
                for j in range(4):
                    nxt[j][i] = l[j]
        else:  # m == "3"
            cols = list(zip(*nxt))
            

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / function_works / mean,rewards / function_works / std,rewards / no_cheating / mean,rewards / no_cheating / std,rewards / strategy_succeeds / mean,rewards / strategy_succeeds / std
1,0.000000,4.000000,0.000000,431.500000,407.000000,456.000000,0.000000,431.500000,407.000000,456.000000,0.000043,1.000000,0.000000,1.000000,0.000000,2.000000,0.000000
2,0.000000,-2.000000,1.414214,518.500000,451.000000,586.000000,0.500000,451.000000,451.000000,451.000000,0.000360,-0.500000,2.121320,0.000000,1.414214,-1.500000,2.121320
3,0.000000,0.500000,4.949748,446.500000,307.000000,586.000000,0.500000,307.000000,307.000000,307.000000,0.000699,-0.500000,2.121320,0.000000,1.414214,1.000000,1.414214
4,0.000000,0.500000,4.949748,555.000000,524.000000,586.000000,0.500000,524.000000,524.000000,524.000000,0.000376,-0.500000,2.121320,0.000000,1.414214,1.000000,1.414214
5,0.000000,4.000000,0.000000,200.000000,92.000000,308.000000,0.000000,200.000000,92.000000,308.000000,0.002642,1.000000,0.000000,1.000000,0.000000,2.000000,0.000000
6,0.000000,0.500000,4.949748,494.500000,403.000000,586.000000,0.500000,403.000000,403.000000,403.000000,0.003704,-0.500000,2.121320,0.000000,1.414214,1.000000,1.414214
7,0.000000,-1.250000,2.474874,556.000000,526.000000,586.000000,0.500000,526.000000,526.000000,526.000000,0.003168,-1.250000,1.060660,0.000000,1.414214,0.000000,0.000000
8,0.000000,-2.000000,1.414214,516.000000,446.000000,586.000000,0.500000,446.000000,446.000000,446.000000,0.025609,-0.500000,2.121320,0.000000,1.414214,-1.500000,2.121320
9,0.000100,4.000000,0.000000,371.500000,254.000000,489.000000,0.000000,371.500000,254.000000,489.000000,0.113482,1.000000,0.000000,1.000000,0.000000,2.000000,0.000000
10,0.000200,0.500000,4.949748,463.500000,341.000000,586.000000,0.500000,341.000000,341.000000,341.000000,0.156487,-0.500000,2.121320,0.000000,1.414214,1.000000,1.414214


Exception = list indices must be integers or slices, not list
def strategy(board):
    size=len(board)
    def add_row(row):
        return [x for x in row if x]
    def compress(row):
        row+=[0]*(size-len(row))
        return row
    def merge(row):
        new=[]
        skip=False
        for i,x in enumerate(row):
            if skip:
                skip=False
                continue
            if i+1<len(row) and row[i]==row[i+1]:
                new.append(x*2)
                skip=True
            else:
                new.append(x)
        return new
    best, best_dir=None,None
    for dir in "0123":
        vec=[]
        if dir=="0":
            for r in board:vec+=add_row(r)
        elif dir=="1":
            for c in range(size):
                vec+=add_row([board[r][c] for r in range(size)])
        elif dir=="2":
            for r in range(size):
                vec+=add_row(board[size-1-r][::-1])
        else:
            for c in range(size):
                

<a name="推理"></a>
# 推论
现在让我们尝试一下我们刚刚训练的模型！

In [35]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize = False,
    add_generation_prompt = True,
    reasoning_effort = "low",
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1.0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-25

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Create a new short 2048 strategy using only native Python code.
You are given a list of list of numbers for the current board state.
Output one action for "0", "1", "2", "3" on what is the optimal next step.
Output your new short function in backticks using the format below:
```python
def strategy(board):
    return "0" # Example
```
All helper functions should be inside def strategy. Only output the short function `strategy`.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to write a short function that, given board (list of lists), outputs optimal move. Use simple heuristic: count possible moves, prioritize. Just code minimal logic.


<a name="保存"></a>
### 保存到 float16 或 `MXFP4`

我们还支持直接保存到`float16`。对于 float16 选择“merged_16bit”，对于 MXFP4 选择“mxfp4”（OpenAI 的 GPT-OSS 原生精度）。我们还允许“lora”适配器作为后备。使用 `push_to_hub_merged` 上传到您的 Hugging Face 帐户！您可以前往 https://huggingface.co/settings/tokens 获取您的个人代币。有关更多部署选项，请参阅 [our docs](https://unsloth.ai/docs/basics/inference-and-deployment)。

In [36]:
# 以 mxfp4 4 位格式合并并推送到集线器
if False:
    model.save_pretrained_merged("openenv_gpt_oss_finetune_4bit", tokenizer, save_method = "mxfp4")
if False:
    model.push_to_hub_merged("repo_id/openenv_gpt_oss_finetune_4bit", tokenizer, token = "YOUR_HF_TOKEN", save_method = "mxfp4")

# 以 16 位合并并推送到集线器
if False:
    model.save_pretrained_merged("openenv_gpt_oss_finetune_16bit", tokenizer, save_method = "merged_16bit")
if False: # 推送至 HF 集线器
    model.push_to_hub_merged("HF_USERNAME/openenv_gpt_oss_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# 我们就完成了！
恭喜您刚刚学会了如何使用 GPT-OSS 进行强化学习！本笔记本中解释了一些高级主题 - 要了解有关 GPT-OSS 和 RL 的更多信息，Unsloth 的 [Reinforcement Learning Guide with GPT-OSS](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning) 中有更多文档

此笔记本和所有 Unsloth 笔记本均已获得 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可。